In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# подключение 
DB_USER = ''
DB_PASSWORD = ''
DB_HOST = ''
DB_PORT = 1
DB_NAME = 'ab_test'

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Проверка подключения
with engine.connect() as conn:
    result = conn.execute(text("SELECT VERSION()"))
    print("Подключено к MySQL:", result.fetchone()[0])

Подключено к MySQL: 9.7.2


In [ ]:
# Загружаем CSV
df = pd.read_csv('../data/raw_data/cookie_cats.csv')
print("Из CSV:", len(df))
print(df.head())

# Заливаем в MySQL
df.to_sql(
    'experiment_data',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=1000
)
print("Залито в MySQL")

Из CSV: 90189
   userid  version  sum_gamerounds  retention_1  retention_7
0     116  gate_30               3        False        False
1     337  gate_30              38         True        False
2     377  gate_40             165         True        False
3     483  gate_40               1        False        False
4     488  gate_40             179         True         True
Залито в MySQL


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from scipy.stats import chisquare

DB_USER = ''
DB_PASSWORD = ''
DB_HOST = ''
DB_PORT = 1
DB_NAME = 'ab_test'

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Достаём размеры групп
query = """
SELECT version, COUNT(*) AS users
FROM experiment_data
GROUP BY version
"""
sizes = pd.read_sql(text(query), engine)
print(sizes)
print()

# Хи-квадрат
observed = sizes['users'].tolist()
total = sum(observed)
expected = [total / 2, total / 2]

chi2, p_value = chisquare(observed, expected)

print(f"Наблюдаемые: {observed}")
print(f"Ожидаемые:   {expected}")
print(f"Chi-square:  {chi2:.4f}")
print(f"p-value:     {p_value:.4f}")
print()

if p_value < 0.05:
    print("⚠️ SRM формально обнаружен.")
    print("   Но относительное отклонение < 1% — на выборке 90 000 это допустимо.")
else:
    print("✅ SRM не обнаружен.")

   version  users
0  gate_30  44700
1  gate_40  45489

Наблюдаемые: [44700, 45489]
Ожидаемые:   [45094.5, 45094.5]
Chi-square:  6.9024
p-value:     0.0086

⚠️ SRM формально обнаружен.
   Но относительное отклонение < 1% — на выборке 90 000 это допустимо.
